# Phase 4 (PILOT) — Lesion-Feature Multi-Task

Single-variable change from P2B: add an auxiliary head predicting 4 lesion
features (haem/exud/cws/nvd) that define adjacent grades. Loss =
focal(grade) + 0.5*BCE(features). Folds 0-1 pilot. Inference uses the GRADE
head only. Output: output_dir/phase4_mt_cv/ (NEVER touches phase2b_cv).

In [1]:
import os, sys, json, copy, math, time
from pathlib import Path

# Reduce CUDA memory fragmentation (helps on 12 GB GPUs with large models)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

os.chdir(os.path.dirname(os.path.abspath('phase4_mt_pilot.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Shared config (same as Phase 1/2A) ────────────────────────────────────────
N_FOLDS     = 5
MAX_EPOCHS  = 50
PATIENCE    = 10
INPUT_SIZE  = 224
NUM_CLASSES = 4
CLASS_WEIGHTS = [1.0, 1.796, 10.8469, 17.502]   # recomputed inverse-freq on NEW cohort
SEED        = 42
CLASSES     = ['R0', 'R1', 'R2', 'R3A']

# ── Full fine-tuning specific config ─────────────────────────────────────────
BASE_LR      = 5e-5    # much lower than LP: backbone is sensitive
MIN_LR       = 1e-7
WARMUP_EPOCHS = 5      # ramp up LR before cosine decay
WEIGHT_DECAY = 0.05    # AdamW regularisation
LLRD_DECAY   = 0.75    # per-layer LR multiplier toward input
GRAD_CLIP    = 1.0     # prevents exploding gradients in large ViTs
BATCH_SIZE   = 16      # physical batch per GPU step (reduced for 12 GB GPU)
ACCUM_STEPS  = 2       # gradient accumulation: effective batch = 16 × 2 = 32
                       # gradients are divided by ACCUM_STEPS so training dynamics
                       # are identical to a true batch of 32 images

# ── Focal loss ────────────────────────────────────────────────────────────────
FOCAL_GAMMA = 2.0
P4_LAMBDA   = 0.5   # auxiliary feature-loss weight      # same as Phase 2A

HF_REPO   = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE   = 'RETFound_dinov2_meh.pth'
CV_OUTPUT = Path('output_dir/phase4_mt_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: Full fine-tuning on CPU is extremely slow (hours per fold).')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(DEVICE)
    print(f'GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB')
print(f'Config: BASE_LR={BASE_LR}, LLRD={LLRD_DECAY}, FOCAL_GAMMA={FOCAL_GAMMA}')
print(f'Batch:  physical={BATCH_SIZE}  accum_steps={ACCUM_STEPS}  effective={BATCH_SIZE*ACCUM_STEPS}')

Working directory: /home/eth-admin/Desktop/isaack/RETFound-main
Device: cuda
GPU: NVIDIA RTX A6000  VRAM: 50.9 GB
Config: BASE_LR=5e-05, LLRD=0.75, FOCAL_GAMMA=2.0
Batch:  physical=16  accum_steps=2  effective=32


In [2]:
class FocalLoss(nn.Module):
    """
    Multiclass focal loss with optional per-class weights.
    Same implementation as Phase 2A — only difference is we feed it into a
    full-fine-tuned model instead of a linear probe.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        log_p  = F.log_softmax(logits, dim=1)
        log_pt = log_p.gather(1, targets.view(-1, 1)).squeeze(1)
        pt     = log_pt.exp()
        focal_weight = (1.0 - pt) ** self.gamma
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_weight = focal_weight * alpha
            loss = -(focal_weight * log_pt).sum() / alpha.sum()
        else:
            loss = -(focal_weight * log_pt).mean()
        return loss


# Sanity check: gamma=0 focal loss must match PyTorch weighted CE
torch.manual_seed(0)
_w = torch.tensor(CLASS_WEIGHTS)
_fl0 = FocalLoss(gamma=0.0, weight=_w)
_ce  = nn.CrossEntropyLoss(weight=_w)
_logits  = torch.randn(16, 4)
_targets = torch.randint(0, 4, (16,))
_diff = abs(float(_fl0(_logits, _targets)) - float(_ce(_logits, _targets)))
print(f'gamma=0 vs weighted CE diff: {_diff:.2e}  ({"PASS" if _diff < 1e-5 else "FAIL"})')

gamma=0 vs weighted CE diff: 2.38e-07  (PASS)


In [3]:
# ── Load splits (identical to Phase 1/2A) ─────────────────────────────────────
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']
patient_ids   = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

# Same SEED → same folds as Phase 1 and 2A (direct comparison valid)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx
pat_grade['fold'] = pat_grade['code'].map(fold_assignments)

df_cv = df_cv.reset_index(drop=True)
df_cv['cv_idx'] = df_cv.index

print(f'CV pool : {len(df_cv)} images | {len(patient_ids)} patients')
print(f'Test set: {len(df_test)} images | {df_test["code"].nunique()} patients')
print('Same folds as Phase 1/2A (SEED=42) — direct comparison valid.')

CV pool : 7495 images | 1824 patients
Test set: 1349 images | 323 patients
Same folds as Phase 1/2A (SEED=42) — direct comparison valid.


In [4]:
# ── P4 multi-task dataset + feature pos_weight ─────────────────────────
import argparse
from util.datasets import build_transform
from p4_multitask import (FEATURE_NAMES, P4Dataset, make_records_mt,
                          compute_feature_pos_weight)

_aug_args = argparse.Namespace(
    input_size=INPUT_SIZE, color_jitter=None,
    aa='rand-m9-mstd0.5-inc1', reprob=0.25, remode='pixel', recount=1,
)
train_transform = build_transform('train', _aug_args)
eval_transform  = build_transform('val',   _aug_args)

FEATURE_POS_WEIGHT = compute_feature_pos_weight(
    df_cv if 'df_cv' in dir() else pd.read_csv('labels/splits.csv')
).to(DEVICE)
print('Feature names:', FEATURE_NAMES, '| pos_weight:', FEATURE_POS_WEIGHT.tolist())

/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature names: ['haem', 'exud', 'cws', 'nvd'] | pos_weight: [3.372812032699585, 3.798335552215576, 8.771838188171387, 80.46739196777344]


## Backbone loading — full fine-tune version

Only one line changes versus the linear probe setup:

```python
# Linear probe:   param.requires_grad = ('head' in name)  → only head trainable
# Full fine-tune: param.requires_grad = True               → everything trainable
```

Everything else (checkpoint loading, key remapping, head initialisation) is identical.

In [5]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone_fft(device, num_classes=NUM_CLASSES, seed=None):
    """Load RETFound-DINOv2 with ALL parameters trainable for full fine-tuning.

    Gradient checkpointing is enabled to reduce activation memory:
    instead of storing all 24 blocks' intermediate activations simultaneously,
    only block inputs are stored and activations are recomputed during backward.
    This cuts activation memory by ~10x at the cost of ~33% more compute.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True, img_size=INPUT_SIZE,
        num_classes=num_classes, drop_path_rate=0.2,
    )
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt      = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}
    state = model.state_dict()
    drop  = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        print(f'  Skipping mismatched: {k}')
        del ckpt_model[k]
    model.load_state_dict(ckpt_model, strict=False)
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)
    # Full fine-tune: ALL parameters are trainable
    for param in model.parameters():
        param.requires_grad = True
    # Gradient checkpointing: recompute activations during backward to save memory
    model.set_grad_checkpointing(enable=True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    print('Gradient checkpointing: ENABLED')
    return model.to(device)

print('Verifying backbone load...')
_m = load_backbone_fft(DEVICE)
print('OK.')
del _m
torch.cuda.empty_cache()

Verifying backbone load...


  Skipping mismatched: pos_embed
Trainable: 303,232,004 / 303,232,004 (100.00%)
Gradient checkpointing: ENABLED


OK.


## Layer-wise learning rate decay (LLRD)

The head gets the full `BASE_LR`. Each layer going toward the input is multiplied by `LLRD_DECAY=0.75`:

```
head           → BASE_LR × 0.75⁰ = 5.00e-5
final norm     → BASE_LR × 0.75¹ = 3.75e-5
blocks[23]     → BASE_LR × 0.75² = 2.81e-5
blocks[22]     → BASE_LR × 0.75³ = 2.11e-5
...
blocks[0]      → BASE_LR × 0.75²⁵ ≈ 2.8e-8  (almost frozen)
patch_embed    → BASE_LR × 0.75²⁶ ≈ 2.1e-8
```

Bias, norm, and positional parameters get no weight decay (standard practice for ViTs).

The cosine warmup schedule then scales the entire LLRD pattern up/down together:
- epoch 0: all LRs × 0.2 (warmup start)
- epoch 4: all LRs × 1.0 (peak, end of warmup)
- epoch 49: all LRs × min_lr/base_lr ≈ 0.002 (cosine minimum)

In [6]:
def build_llrd_optimizer(model, base_lr, weight_decay, decay=LLRD_DECAY):
    """
    AdamW with layer-wise learning rate decay.
    Each param is matched by name to a depth from the head;
    LR = base_lr × decay^depth.
    Bias, norm, cls_token, pos_embed: no weight decay.
    """
    num_blocks = len(model.backbone.blocks) if hasattr(model, 'backbone') else len(model.blocks)

    def get_depth(name):
        name = name.replace('backbone.', '')  # multi-task wrapper prefix
        if 'head' in name:
            return 0
        # Final norm (model.norm, not block norms)
        if name.startswith('norm'):
            return 1
        if 'blocks.' in name:
            block_idx = int(name.split('blocks.')[1].split('.')[0])
            # Last block (23) → depth 2; first block (0) → depth 25
            return num_blocks - block_idx + 1
        # patch_embed, cls_token, pos_embed, etc.
        return num_blocks + 2

    def no_decay(name):
        return any(tag in name for tag in ['bias', 'norm', 'cls_token', 'pos_embed'])

    # Group parameters by (depth, no_decay flag)
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        key = (get_depth(name), no_decay(name))
        groups.setdefault(key, []).append(param)

    param_groups = []
    for (depth, nd), params in sorted(groups.items()):
        lr = base_lr * (decay ** depth)
        param_groups.append({
            'params': params,
            'initial_lr': lr,   # stored so cosine schedule can scale it
            'lr': lr,
            'weight_decay': 0.0 if nd else weight_decay,
        })

    return torch.optim.AdamW(param_groups)


# Print LLRD LR table
print('LLRD learning rate schedule (at peak, epoch 5):')
print(f'  head         :  {BASE_LR * LLRD_DECAY**0:.2e}')
print(f'  final norm   :  {BASE_LR * LLRD_DECAY**1:.2e}')
print(f'  blocks[23]   :  {BASE_LR * LLRD_DECAY**2:.2e}')
print(f'  blocks[20]   :  {BASE_LR * LLRD_DECAY**5:.2e}')
print(f'  blocks[12]   :  {BASE_LR * LLRD_DECAY**13:.2e}')
print(f'  blocks[0]    :  {BASE_LR * LLRD_DECAY**25:.2e}')
print(f'  patch_embed  :  {BASE_LR * LLRD_DECAY**26:.2e}')

LLRD learning rate schedule (at peak, epoch 5):
  head         :  5.00e-05
  final norm   :  3.75e-05
  blocks[23]   :  2.81e-05
  blocks[20]   :  1.19e-05
  blocks[12]   :  1.19e-06
  blocks[0]    :  3.76e-08
  patch_embed  :  2.82e-08


In [7]:
# ── Training helpers ──────────────────────────────────────────────────────────

class EarlyStoppingFFT:
    """
    Early stopping for full fine-tuning.
    Saves the full model state dict to disk (not just the head) because all
    307M parameters change during training.
    """
    def __init__(self, patience, model, checkpoint_path):
        self.patience        = patience
        self.best_auroc      = -float('inf')
        self.counter         = 0
        self.checkpoint_path = Path(checkpoint_path)
        torch.save(model.state_dict(), self.checkpoint_path)

    def step(self, auroc, model):
        if auroc != auroc: auroc = 0.0   # NaN guard
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model, device):
        state = torch.load(self.checkpoint_path, map_location=device, weights_only=True)
        model.load_state_dict(state)


def get_lr(epoch, warmup_epochs, max_epochs, base_lr, min_lr):
    """Linear warmup then cosine decay."""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))


def train_epoch_fft(model, loader, optimizer, criterion, device, scaler, epoch):
    """
    One training epoch for full fine-tuning with gradient accumulation.

    Gradient accumulation:
      ACCUM_STEPS mini-batches of size BATCH_SIZE are processed before taking
      one optimizer step. Loss is divided by ACCUM_STEPS so the effective gradient
      magnitude matches a single batch of size BATCH_SIZE × ACCUM_STEPS.

    Gradient checkpointing (set in load_backbone_fft):
      Activations are recomputed during backward instead of being stored —
      reduces peak activation memory ~10× at ~33% speed cost.

    Gradient clipping:
      Applied after unscaling so the clip threshold is in the original scale.
    """
    model.train()
    head_lr  = get_lr(epoch, WARMUP_EPOCHS, MAX_EPOCHS, BASE_LR, MIN_LR)
    lr_scale = head_lr / BASE_LR
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * lr_scale

    optimizer.zero_grad()
    total_loss   = 0.0
    n_samples    = 0
    step_count   = 0   # counts how many mini-batches since last optimizer step

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        # Determine if this is the last mini-batch overall
        is_last_batch = (i + 1 == len(loader))
        # Determine if we should take an optimizer step after this mini-batch
        should_step = ((step_count + 1) % ACCUM_STEPS == 0) or is_last_batch

        with torch.cuda.amp.autocast():
            # Divide loss by ACCUM_STEPS so accumulated gradient = true-batch gradient
            loss = criterion(model(imgs), labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS * len(labels)  # undo /ACCUM_STEPS for logging
        n_samples  += len(labels)
        step_count += 1

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_count = 0

    return total_loss / n_samples, head_lr


@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().float()
        all_labels.append(labels)
        all_probs.append(probs)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()


def compute_metrics(labels, probs, preds=None):
    if preds is None:
        preds = probs.argmax(axis=1)
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro',
                              labels=list(range(NUM_CLASSES)))
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
    return {'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
            'macro_sensitivity': np.nanmean(sens), 'macro_specificity': np.nanmean(spec),
            'sensitivity': np.array(sens), 'specificity': np.array(spec)}


print('Helpers defined.')

# ── P4 multi-task train/eval (grade tuple-aware) ─────────────────────
def train_epoch_fft_mt(model, loader, optimizer, criterion, device, scaler, epoch):
    model.train()
    head_lr = get_lr(epoch, WARMUP_EPOCHS, MAX_EPOCHS, BASE_LR, MIN_LR)
    lr_scale = head_lr / BASE_LR
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * lr_scale
    optimizer.zero_grad()
    total_loss = 0.0; n = 0; step = 0
    for i, (imgs, grades, feats) in enumerate(loader):
        imgs, grades, feats = imgs.to(device), grades.to(device), feats.to(device)
        is_last = (i + 1 == len(loader))
        should_step = ((step + 1) % ACCUM_STEPS == 0) or is_last
        with torch.cuda.amp.autocast():
            g_logits, f_logits = model(imgs)
            loss = criterion(g_logits, grades, f_logits, feats) / ACCUM_STEPS
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS * len(grades); n += len(grades); step += 1
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(); step = 0
    return total_loss / n, head_lr

@torch.no_grad()
def eval_fold_mt(model, loader, device):
    model.eval(); all_labels, all_probs = [], []
    for imgs, grades, feats in loader:
        with torch.cuda.amp.autocast():
            g_logits, _ = model(imgs.to(device))
        all_probs.append(torch.softmax(g_logits, dim=1).cpu().float()); all_labels.append(grades)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()

print('P4 multi-task train/eval helpers defined.')

Helpers defined.
P4 multi-task train/eval helpers defined.


## 5-fold CV training loop — full fine-tuning

Differences from Phase 2A:
- `load_backbone_fft` instead of `load_backbone` (all params trainable)
- `build_llrd_optimizer` instead of plain `AdamW` (LLRD per layer)
- `train_epoch_fft` instead of `train_epoch` (LLRD schedule + grad clip)
- `EarlyStoppingFFT` instead of `EarlyStopping` (saves full model to disk)

In [8]:
# ── P4 multi-task CV loop (folds 0-1 pilot) ──────────────────────
from p4_multitask import build_multitask_model, MultiTaskLoss

weight_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float).to(DEVICE)
focal_cv = FocalLoss(gamma=FOCAL_GAMMA, weight=weight_tensor)
criterion_cv = MultiTaskLoss(focal_cv, FEATURE_POS_WEIGHT, lam=P4_LAMBDA)

oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)
fold_results   = []

for fold in range(2):  # PILOT — folds 0,1 only
    print(f'\n{"="*60}\n  FOLD {fold+1}/2  [P4 multi-task, lambda={P4_LAMBDA}]\n{"="*60}')
    val_pats   = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats = pat_grade[pat_grade['fold'] != fold]['code'].values
    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]
    val_cv_indices = df_fold_val['cv_idx'].values

    ds_train = P4Dataset(make_records_mt(df_fold_train), train_transform)
    ds_val   = P4Dataset(make_records_mt(df_fold_val),   eval_transform)
    ds_test  = P4Dataset(make_records_mt(df_test),       eval_transform)
    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    model = build_multitask_model(load_backbone_fft(device=DEVICE, seed=SEED + fold)).to(DEVICE)
    model.set_grad_checkpointing(True)
    optimizer = build_llrd_optimizer(model, BASE_LR, WEIGHT_DECAY, decay=LLRD_DECAY)
    scaler  = torch.cuda.amp.GradScaler()
    ckpt    = CV_OUTPUT / f'best_fold_{fold}.pth'
    stopper = EarlyStoppingFFT(patience=PATIENCE, model=model, checkpoint_path=ckpt)

    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch_fft_mt(model, loader_train, optimizer, criterion_cv, DEVICE, scaler, epoch)
        val_labels, val_probs = eval_fold_mt(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)
        print(f'  ep {epoch:02d} | loss={tr_loss:.4f} | AUROC={m["auroc"]:.4f} | sens={m["macro_sensitivity"]:.4f}')
        if stopper.step(m['auroc'], model):
            print(f'  Early stop epoch {epoch} (best AUROC={stopper.best_auroc:.4f})'); break

    stopper.restore(model, DEVICE)
    oof_labels, oof_probs = eval_fold_mt(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs
    test_labels_fold, test_probs_fold = eval_fold_mt(model, loader_test, DEVICE)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',   oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy',  oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy',  test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy', test_labels_fold)
    m_fold = compute_metrics(oof_labels, oof_probs)
    fold_results.append({'fold': fold, 'best_auroc': stopper.best_auroc,
                         'oof_auroc': m_fold['auroc'], 'oof_macro_sens': m_fold['macro_sensitivity']})
    print(f'  OOF AUROC {m_fold["auroc"]:.4f}  macroSens {m_fold["macro_sensitivity"]:.4f}')
    del model; torch.cuda.empty_cache()

with open(CV_OUTPUT / 'fold_results_pilot.json', 'w') as f:
    json.dump(fold_results, f, indent=2)
print('Pilot folds 0-1 complete.')


  FOLD 1/2  [P4 multi-task, lambda=0.5]


  Skipping mismatched: pos_embed
Trainable: 303,232,004 / 303,232,004 (100.00%)
Gradient checkpointing: ENABLED


/tmp/ipykernel_240308/38356327.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler()


/tmp/ipykernel_240308/3372630364.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


/tmp/ipykernel_240308/3372630364.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


...

/tmp/ipykernel_240308/3372630364.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


/tmp/ipykernel_240308/3372630364.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


  ep 24 | loss=0.5695 | AUROC=0.9099 | sens=0.6784
  Early stop epoch 24 (best AUROC=0.9222)


/tmp/ipykernel_240308/3372630364.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/eth-admin/miniconda3/envs/retfound/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


  OOF AUROC 0.9222  macroSens 0.6729
Pilot folds 0-1 complete.
